# Forward-Looking Backtest: Calendar-Time Cutoff Comparison

This notebook evaluates the **predictive power** of four competing-risk models by
projecting cash flows forward from a calendar-time cutoff date and comparing against
realized outcomes.

**No data leakage**: All models are **retrained** using only
data available before the cutoff. Predictions use only cutoff-date features.

## Models

| Model | Type | Training data | Prediction input |
|-------|------|---------------|------------------|
| **Aalen-Johansen** | Nonparametric CIF | Pre-cutoff loan histories | Population-level (no features) |
| **Cox TV** | Semi-parametric | Pre-cutoff loan-month panel | Frozen macro + deterministic bal_repaid, t_act_12m |
| **RSF** | ML ensemble | Pre-cutoff terminal observations | Snapshot features at cutoff |
| **DeepHit** | Neural network | Pre-cutoff terminal observations | Snapshot features at cutoff |

## Feature handling after cutoff

- **Evolve deterministically**: `bal_repaid` (amortization), `t_act_12m` (min(12, age))
- **Freeze at cutoff value**: All 12 macro features, `t_del_30d_12m`, `t_del_60d_12m`
- **Always static**: `int_rate`, `orig_upb`/`log_upb`, `fico_score`, `dti_r`, `ltv_r`

## Cutoff date

2021-06

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import torch
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from lifelines import AalenJohansenFitter, CoxTimeVaryingFitter
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

import sys
sys.path.insert(0, '..')

from src.alm.baseline_hazard import extract_baseline_hazards_both
from src.alm.cash_flow_engine import CashFlowConfig, MortgageCashFlowEngine
from src.alm.rsf_cash_flow_engine import RSFCashFlowConfig, RSFCashFlowEngine
from src.alm.deephit_cash_flow_engine import DeepHitCashFlowConfig, DeepHitCashFlowEngine

sns.set_style('whitegrid')
%matplotlib inline

DATA_DIR = Path('../data/processed')
EXTERNAL_DIR = Path('../data/external')
MODELS_DIR = Path('../models')
FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)

print('Imports complete.')

Imports complete.


In [3]:
# Cutoff date and evaluation horizons
CUTOFF_DATES = [
    pd.Period('2021-06', 'M'),
]
HORIZONS = [12, 24, 36, 60, 120, 240, 360]  # months post-cutoff to evaluate
MAX_HORIZON = 360  # full remaining maturity
LGD = 0.25
DISCOUNT_RATE = 0.04  # annual discount rate for PV calculations

print(f'Cutoff dates: {[str(c) for c in CUTOFF_DATES]}')
print(f'Evaluation horizons: {HORIZONS} months')
print(f'Max horizon: {MAX_HORIZON} months ({MAX_HORIZON/12:.0f} years)')
print(f'Discount rate: {DISCOUNT_RATE:.1%}')

Cutoff dates: ['2020-06']
Evaluation horizons: [12, 24, 36, 60, 120, 240, 360] months
Max horizon: 360 months (30 years)
Discount rate: 4.0%


In [ ]:
# Feature definitions (matching notebooks 05 and 07)
COX_FEATURE_NAMES = [
    'int_rate', 'orig_upb', 'fico_score', 'dti_r', 'ltv_r',
    'bal_repaid', 't_act_12m', 't_del_30d_12m', 't_del_60d_12m',
    'hpi_st_d_t_o', 'ppi_c_FRMA', 'TB10Y_d_t_o', 'FRMA30Y_d_t_o',
    'ppi_o_FRMA', 'hpi_st_log12m', 'hpi_r_st_us', 'st_unemp_r12m',
    'st_unemp_r3m', 'TB10Y_r12m', 'T10Y3MM', 'T10Y3MM_r12m',
]

# RSF and DeepHit use log_upb instead of orig_upb and bal_repaid_lag1 instead of bal_repaid
RSF_FEATURE_COLS = [
    'int_rate', 'log_upb', 'fico_score', 'dti_r', 'ltv_r',
    'bal_repaid_lag1', 't_act_12m', 't_del_30d_12m', 't_del_60d_12m',
    'hpi_st_d_t_o', 'ppi_c_FRMA', 'TB10Y_d_t_o', 'FRMA30Y_d_t_o',
    'ppi_o_FRMA', 'hpi_st_log12m', 'hpi_r_st_us', 'st_unemp_r12m',
    'st_unemp_r3m', 'TB10Y_r12m', 'T10Y3MM', 'T10Y3MM_r12m',
]

# DeepHit uses same features as RSF
DEEPHIT_FEATURE_COLS = RSF_FEATURE_COLS.copy()

# RSF hyperparameters (matching notebook 07), based on Blumenstock
RSF_PARAMS = {
    'n_estimators': 100,
    'max_depth': 10,
    'min_samples_split': 20,
    'min_samples_leaf': 30,
    'max_features': 2,
    'n_jobs': -1,
    'random_state': 42,
}

# Cox penalizer (matching notebook 05)
COX_PENALIZER = 0.01

print(f'Cox features ({len(COX_FEATURE_NAMES)})')
print(f'RSF features ({len(RSF_FEATURE_COLS)})')
print(f'DeepHit features ({len(DEEPHIT_FEATURE_COLS)})')
print(f'RSF params: {RSF_PARAMS}')

Cox features (21)
RSF features (21)
RSF params: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 20, 'min_samples_leaf': 30, 'max_features': 2, 'n_jobs': -1, 'random_state': 42}


In [ ]:
# Load panel and supporting data
panel_df = pd.read_parquet(DATA_DIR / 'loan_month_panel.parquet')
surv_df = pd.read_parquet(DATA_DIR / 'survival_data_blumenstock.parquet')

# Join orig_loan_term and origination-time macro onto panel
orig_cols = ['loan_sequence_number', 'orig_loan_term', 'orig_MORTGAGE30US', 'orig_DGS10', 'orig_state_hpi']
orig_info = surv_df[orig_cols].drop_duplicates(subset=['loan_sequence_number'])

print(f'Panel: {len(panel_df):,} loan-months, {panel_df["loan_sequence_number"].nunique():,} loans')
print(f'Date range: {panel_df["year_month"].min()} to {panel_df["year_month"].max()}')

---

## Helper Functions

1. **`get_active_loans_at_cutoff`**: Identify loans still active at the cutoff date
2. **`train_cox_models`**: Train Cox TV models on pre-cutoff panel data
3. **`train_rsf_models`**: Train RSF models on pre-cutoff terminal observations
4. **`train_deephit_models`**: Train DeepHit on pre-cutoff terminal observations
5. **`build_frozen_cox_matrix`**: Build 3D covariate matrix with frozen macro
6. **`build_rsf_snapshot`**: Build 2D feature matrix for RSF
7. **`compute_realized`**: Extract realized CIF from post-cutoff panel
8. **`fit_conditional_aj`**: Fit Aalen-Johansen CIF on pre-cutoff data

In [ ]:
def get_active_loans_at_cutoff(panel_df, orig_info, cutoff):
    """
    Get all loans active at the cutoff date with their feature values.
    A loan is active if it has an observation at cutoff with event == 0.

    For bal_repaid_lag1, we use the previous month's bal_repaid to match
    the lag-1 convention used in RSF training.
    """
    cutoff_rows = panel_df[
        (panel_df['year_month'] == cutoff) & (panel_df['event'] == 0)
    ].copy()

    # Join origination info
    cutoff_rows = cutoff_rows.merge(orig_info, on='loan_sequence_number', how='left')
    cutoff_rows['orig_loan_term'] = cutoff_rows['orig_loan_term'].fillna(360).astype(int)
    cutoff_rows['current_loan_age'] = cutoff_rows['loan_age'].astype(int)
    cutoff_rows['log_upb'] = np.log(cutoff_rows['orig_upb'].astype(float))

    # bal_repaid_lag1: use previous month's bal_repaid (matching RSF training convention)
    prev_month = cutoff - 1
    prev_rows = panel_df[
        (panel_df['year_month'] == prev_month)
    ][['loan_sequence_number', 'bal_repaid']].rename(
        columns={'bal_repaid': 'bal_repaid_lag1'}
    )
    cutoff_rows = cutoff_rows.merge(prev_rows, on='loan_sequence_number', how='left')
    # Fallback: if no previous month row exists, use current bal_repaid
    cutoff_rows['bal_repaid_lag1'] = cutoff_rows['bal_repaid_lag1'].fillna(
        cutoff_rows['bal_repaid']
    ).astype(float)

    return cutoff_rows

In [ ]:
def train_cox_models(panel_df, cutoff, feature_names, penalizer=0.01):
    """
    Train cause-specific Cox TV models on all panel data up to the cutoff.

    Returns (cox_engine, feature_list, n_train_loans, n_train_months,
             n_prepay, n_default, max_observed_month)
    """
    # Filter to pre-cutoff data only
    train_panel = panel_df[panel_df['year_month'] <= cutoff].copy()
    train_panel = train_panel.dropna(subset=feature_names).copy()

    n_loans = train_panel['loan_sequence_number'].nunique()
    n_months = len(train_panel)

    # Prepayment: event = 1 for prepay, 0 for default or censored
    train_panel['event_prepay'] = (
        (train_panel['event'] == 1) & (train_panel['event_code'] == 1)
    ).astype(int)

    # Default: event = 1 for default, 0 for prepay or censored
    train_panel['event_default'] = (
        (train_panel['event'] == 1) & (train_panel['event_code'] == 2)
    ).astype(int)

    cox_cols_p = ['loan_sequence_number', 'start', 'stop', 'event_prepay'] + feature_names
    cox_cols_d = ['loan_sequence_number', 'start', 'stop', 'event_default'] + feature_names

    # Fit prepay model
    ctv_prepay = CoxTimeVaryingFitter(penalizer=penalizer)
    ctv_prepay.fit(
        train_panel[cox_cols_p],
        id_col='loan_sequence_number',
        start_col='start',
        stop_col='stop',
        event_col='event_prepay',
        show_progress=False,
    )

    # Fit default model
    ctv_default = CoxTimeVaryingFitter(penalizer=penalizer)
    ctv_default.fit(
        train_panel[cox_cols_d],
        id_col='loan_sequence_number',
        start_col='start',
        stop_col='stop',
        event_col='event_default',
        show_progress=False,
    )

    # Max observed month in baseline hazard (before extrapolation)
    max_obs_p = int(ctv_prepay.baseline_cumulative_hazard_.index.max())
    max_obs_d = int(ctv_default.baseline_cumulative_hazard_.index.max())
    max_observed_month = min(max_obs_p, max_obs_d)

    # Build engine
    h0_prepay, h0_default = extract_baseline_hazards_both(ctv_prepay, ctv_default)
    beta_prepay = ctv_prepay.params_.values.astype(np.float64)
    beta_default = ctv_default.params_.values.astype(np.float64)

    engine = MortgageCashFlowEngine(
        h0_prepay, h0_default, beta_prepay, beta_default,
        config=CashFlowConfig(lgd=LGD, projection_horizon=MAX_HORIZON),
    )

    n_prepay = train_panel['event_prepay'].sum()
    n_default = train_panel['event_default'].sum()

    return engine, ctv_prepay.params_.index.tolist(), n_loans, n_months, n_prepay, n_default, max_observed_month

In [ ]:
def train_rsf_models(panel_df, cutoff, rsf_feature_cols, rsf_params):
    """
    Train cause-specific RSF models on terminal observations from pre-cutoff data.

    Returns (rsf_engine, n_train_loans, n_prepay, n_default, domain_max)
    """
    # Get pre-cutoff panel
    before = panel_df[panel_df['year_month'] <= cutoff].copy()
    before = before.sort_values(['loan_sequence_number', 'loan_age'])

    # Last observation per loan (terminal or censored-at-cutoff)
    terminal_df = before.groupby('loan_sequence_number').last().reset_index()

    # Lag bal_repaid to avoid leakage (matching notebook 07)
    def get_lagged_bal_repaid(group):
        if len(group) >= 2:
            return group['bal_repaid'].iloc[-2]
        else:
            return group['bal_repaid'].iloc[-1]

    bal_repaid_lag = before.groupby('loan_sequence_number').apply(get_lagged_bal_repaid)
    terminal_df['bal_repaid_lag1'] = terminal_df['loan_sequence_number'].map(bal_repaid_lag)

    # Log transform UPB
    terminal_df['log_upb'] = np.log(terminal_df['orig_upb'].astype(float))

    # Drop NaN features
    terminal_df = terminal_df.dropna(subset=rsf_feature_cols).copy()

    X_train = terminal_df[rsf_feature_cols].values
    duration = terminal_df['loan_age'].values.astype(float)
    event_code = terminal_df['event_code'].values

    n_loans = len(terminal_df)
    n_prepay = int((event_code == 1).sum())
    n_default = int((event_code == 2).sum())

    # Prepayment RSF
    y_prepay = Surv.from_arrays(event_code == 1, duration)
    rsf_p = RandomSurvivalForest(**rsf_params)
    rsf_p.fit(X_train, y_prepay)

    # Default RSF
    y_default = Surv.from_arrays(event_code == 2, duration)
    rsf_d = RandomSurvivalForest(**rsf_params)
    rsf_d.fit(X_train, y_default)

    # RSF domain max: max time the survival curves can be evaluated
    # Query from a single prediction to get the domain
    sample_surv_p = rsf_p.predict_survival_function(X_train[:1])
    sample_surv_d = rsf_d.predict_survival_function(X_train[:1])
    domain_max = int(min(sample_surv_p[0].domain[1], sample_surv_d[0].domain[1]))

    engine = RSFCashFlowEngine(
        rsf_p, rsf_d,
        config=RSFCashFlowConfig(lgd=LGD, projection_horizon=MAX_HORIZON),
    )

    return engine, n_loans, n_prepay, n_default, domain_max

In [ ]:
# === DeepHit Model Architecture ===

class DeepHitNetwork(torch.nn.Module):
    """
    DeepHit neural network for competing risks (Lee et al., 2018).

    Architecture:
    - Shared FFN: 1 layer, 300 nodes
    - Skip connection: concatenate raw input with shared output
    - Cause-specific heads: 2 heads (prepay, default), each 3 layers, 100 nodes
    - Activation: ReLU
    - Joint softmax over (time, cause) for PMF
    """
    def __init__(
        self,
        in_features: int,
        num_time_bins: int,
        num_causes: int = 2,
        shared_layers: int = 1,
        shared_nodes: int = 300,
        head_layers: int = 3,
        head_nodes: int = 100,
        dropout: float = 0.6,
        batch_norm: bool = True,
    ):
        super().__init__()
        self.in_features = in_features
        self.num_time_bins = num_time_bins
        self.num_causes = num_causes

        shared = []
        prev_dim = in_features
        for _ in range(shared_layers):
            shared.append(torch.nn.Linear(prev_dim, shared_nodes))
            if batch_norm:
                shared.append(torch.nn.BatchNorm1d(shared_nodes))
            shared.append(torch.nn.ReLU())
            shared.append(torch.nn.Dropout(dropout))
            prev_dim = shared_nodes
        self.shared = torch.nn.Sequential(*shared)

        # Each head receives concat(x, shared_out) per Lee et al. (2018)
        self.heads = torch.nn.ModuleList()
        for _ in range(num_causes):
            head = []
            prev_dim = shared_nodes + in_features  # concatenation
            for _ in range(head_layers):
                head.append(torch.nn.Linear(prev_dim, head_nodes))
                if batch_norm:
                    head.append(torch.nn.BatchNorm1d(head_nodes))
                head.append(torch.nn.ReLU())
                head.append(torch.nn.Dropout(dropout))
                prev_dim = head_nodes
            head.append(torch.nn.Linear(prev_dim, num_time_bins))
            self.heads.append(torch.nn.Sequential(*head))

    def forward(self, x):
        batch_size = x.shape[0]
        shared_out = self.shared(x)
        # Skip connection: concatenate raw input with shared output
        combined = torch.cat([x, shared_out], dim=1)
        head_outputs = [head(combined) for head in self.heads]
        logits = torch.stack(head_outputs, dim=1)
        logits_flat = logits.view(batch_size, -1)
        pmf_flat = torch.softmax(logits_flat, dim=-1)
        pmf = pmf_flat.view(batch_size, self.num_causes, self.num_time_bins)
        return pmf

    def predict_cif(self, x):
        pmf = self.forward(x)
        return torch.cumsum(pmf, dim=-1)

    def predict_survival(self, x):
        cif = self.predict_cif(x)
        return 1 - cif.sum(dim=1)


class DeepHitLoss(torch.nn.Module):
    """DeepHit loss: NLL + ranking loss for competing risks."""
    def __init__(self, alpha=1.0, sigma=0.1):
        super().__init__()
        self.alpha = alpha
        self.sigma = sigma

    def forward(self, pmf, durations, events, time_bins):
        batch_size = pmf.shape[0]
        num_causes = pmf.shape[1]
        num_bins = pmf.shape[2]
        device = pmf.device
        eps = 1e-7

        bin_indices = torch.bucketize(durations, time_bins[1:])
        bin_indices = torch.clamp(bin_indices, 0, num_bins - 1)

        cif = torch.cumsum(pmf, dim=-1)
        total_cif = cif.sum(dim=1)
        survival = torch.clamp(1 - total_cif, min=0.0)
        survival_at_time = survival[torch.arange(batch_size, device=device), bin_indices]

        cause_indices = torch.clamp((events - 1).long(), 0, num_causes - 1)
        pmf_at_event = pmf[torch.arange(batch_size, device=device), cause_indices, bin_indices]

        is_censored = (events == 0).float()
        is_uncensored = (events > 0).float()

        nll_loss = (
            -torch.log(pmf_at_event + eps) * is_uncensored
            + -torch.log(survival_at_time + eps) * is_censored
        ).mean()

        if self.alpha > 0 and is_uncensored.sum() > 0:
            ranking_loss = self._compute_ranking_loss(cif, bin_indices, events, num_causes)
        else:
            ranking_loss = torch.tensor(0.0, device=device)

        return nll_loss + self.alpha * ranking_loss, nll_loss, ranking_loss

    def _compute_ranking_loss(self, cif, bin_indices, events, num_causes):
        """Vectorized ranking loss over all valid pairs in mini-batch (Lee et al., 2018)."""
        device = cif.device
        ranking_loss = torch.tensor(0.0, device=device)
        n_pairs = 0
        for k in range(num_causes):
            event_code = k + 1
            cause_mask = (events == event_code)
            idx_i = torch.where(cause_mask)[0]
            if len(idx_i) == 0:
                continue
            t_i = bin_indices[idx_i]
            t_j = bin_indices
            valid_pairs = t_j.unsqueeze(0) > t_i.unsqueeze(1)
            if not valid_pairs.any():
                continue
            cif_i = cif[idx_i, k, t_i]
            cif_j = cif[:, k, :][:, t_i].T
            diff = cif_j - cif_i.unsqueeze(1)
            pair_loss = torch.exp(diff / self.sigma) * valid_pairs.float()
            ranking_loss = ranking_loss + pair_loss.sum()
            n_pairs += valid_pairs.sum().item()
        if n_pairs > 0:
            ranking_loss = ranking_loss / n_pairs
        return ranking_loss



class JointDeepHitCashFlowEngine:
    """
    Cash flow engine that uses a joint DeepHit model directly.

    Unlike the base DeepHitCashFlowEngine (which expects separate cause-specific
    nets and re-normalizes each via softmax), this engine extracts cause-specific
    hazards from the joint model's CIF without renormalization:

        h_k(t) = [CIF_k(t) - CIF_k(t-1)] / S(t-1)

    where S(t) = 1 - CIF_prepay(t) - CIF_default(t) is the overall survival.
    """

    def __init__(self, joint_model, scaler, time_points, config=None, device='cpu'):
        self.joint_model = joint_model
        self.scaler = scaler
        self.cuts = time_points.astype(np.float64)
        self.config = config or DeepHitCashFlowConfig()
        self.device = device
        self.domain_max = int(self.cuts[-1])

    def predict_hazards(self, X, current_age, T):
        """
        Extract cause-specific hazards from the joint model's CIF.

        h_k(t) = f_k(t) / S(t-1)
        where f_k(t) = CIF_k(t) - CIF_k(t-1) and S(t) = 1 - sum_k CIF_k(t).
        """
        N = len(X)
        X_scaled = self.scaler.transform(X).astype(np.float32)
        max_age = min(int(current_age.max()) + T + 1, self.domain_max)

        # Get joint CIF from model (no renormalization)
        with torch.no_grad():
            x_t = torch.tensor(X_scaled, dtype=torch.float32).to(self.device)
            cif = self.joint_model.predict_cif(x_t).cpu().numpy()  # (N, 2, n_bins)

        # Interpolate CIF for each cause to monthly grid
        months = np.arange(max_age + 1, dtype=np.float64)
        cif_p = np.zeros((N, max_age + 1), dtype=np.float64)
        cif_d = np.zeros((N, max_age + 1), dtype=np.float64)
        for i in range(N):
            cif_p[i] = np.interp(months, self.cuts, cif[i, 0], left=0.0)
            cif_d[i] = np.interp(months, self.cuts, cif[i, 1], left=0.0)

        # Build age index arrays
        t_arr = np.arange(T)[None, :]
        age_curr = (current_age[:, None].astype(int) + t_arr + 1)
        age_prev = age_curr - 1
        age_curr = np.clip(age_curr, 0, max_age)
        age_prev = np.clip(age_prev, 0, max_age)
        row_idx = np.arange(N)[:, None]

        # Sub-densities: f_k(t) = CIF_k(t) - CIF_k(t-1)
        f_p = np.maximum(cif_p[row_idx, age_curr] - cif_p[row_idx, age_prev], 0.0)
        f_d = np.maximum(cif_d[row_idx, age_curr] - cif_d[row_idx, age_prev], 0.0)

        # Overall survival: S(t-1) = 1 - CIF_prepay(t-1) - CIF_default(t-1)
        S_prev = np.maximum(
            1.0 - cif_p[row_idx, age_prev] - cif_d[row_idx, age_prev], 1e-10
        )

        # Cause-specific hazards: h_k(t) = f_k(t) / S(t-1)
        h_prepay = np.clip(f_p / S_prev, 0.0, 1.0)
        h_default = np.clip(f_d / S_prev, 0.0, 1.0)

        return h_prepay, h_default

    def project_cash_flows(self, loans_df, X):
        """Project cash flows for a portfolio of loans."""
        N = len(loans_df)
        T = self.config.projection_horizon
        cfg = self.config

        int_rate = loans_df['int_rate'].values.astype(np.float64)
        orig_upb = loans_df['orig_upb'].values.astype(np.float64)
        term = loans_df['orig_loan_term'].values.astype(np.float64)
        current_age = loans_df['current_loan_age'].values.astype(np.float64)

        results = {
            k: np.zeros((N, T), dtype=np.float64)
            for k in ['interest', 'scheduled_principal', 'prepayment', 'recovery',
                       'loss', 'total_cf', 'survival', 'f_prepay', 'f_default',
                       'upb_schedule']
        }

        for b_start in range(0, N, cfg.batch_size):
            b_end = min(b_start + cfg.batch_size, N)
            sl = slice(b_start, b_end)
            batch = self._project_batch(
                int_rate[sl], orig_upb[sl], term[sl], current_age[sl],
                X[b_start:b_end], T,
            )
            for key in results:
                results[key][sl] = batch[key]

        return results

    def _project_batch(self, int_rate, orig_upb, term, current_age, X_batch, T):
        """Project cash flows for a batch of loans."""
        n = len(int_rate)
        cfg = self.config

        # --- 1. AMORTIZATION SCHEDULE ---
        monthly_rate = int_rate / 100.0 / 12.0
        payment = np.where(
            monthly_rate > 0,
            orig_upb * monthly_rate / (1.0 - (1.0 + monthly_rate) ** (-term)),
            orig_upb / term,
        )
        factor_start = (1.0 + monthly_rate) ** current_age
        upb_start = np.where(
            monthly_rate > 0,
            orig_upb * factor_start - payment * (factor_start - 1.0) / monthly_rate,
            orig_upb - payment * current_age,
        )
        upb_start = np.maximum(upb_start, 0.0)

        upb = np.zeros((n, T), dtype=np.float64)
        interest = np.zeros((n, T), dtype=np.float64)
        sched_principal = np.zeros((n, T), dtype=np.float64)

        prev_upb = upb_start.copy()
        for t in range(T):
            total_age = current_age + t + 1
            active = total_age <= term
            int_t = prev_upb * monthly_rate * active
            prin_t = np.minimum(payment - int_t, prev_upb) * active
            prin_t = np.maximum(prin_t, 0.0)
            new_upb = np.maximum(prev_upb - prin_t, 0.0)
            interest[:, t] = int_t
            sched_principal[:, t] = prin_t
            upb[:, t] = new_upb
            prev_upb = new_upb

        # --- 2. CAUSE-SPECIFIC HAZARDS FROM JOINT DEEPHIT ---
        h_prepay, h_default = self.predict_hazards(X_batch, current_age, T)

        # Clip total hazard
        h_sum = h_prepay + h_default
        scale = np.where(h_sum > cfg.max_hazard_total, cfg.max_hazard_total / h_sum, 1.0)
        h_prepay = h_prepay * scale
        h_default = h_default * scale

        # --- 3. SURVIVAL AND SUB-DENSITIES ---
        survival = np.zeros((n, T), dtype=np.float64)
        f_prepay = np.zeros((n, T), dtype=np.float64)
        f_default = np.zeros((n, T), dtype=np.float64)

        s_prev = np.ones(n, dtype=np.float64)
        for t in range(T):
            f_prepay[:, t] = h_prepay[:, t] * s_prev
            f_default[:, t] = h_default[:, t] * s_prev
            s_prev = s_prev * (1.0 - h_prepay[:, t] - h_default[:, t])
            survival[:, t] = s_prev

        # --- 4. EXPECTED CASH FLOWS ---
        s_start = np.ones((n, T), dtype=np.float64)
        s_start[:, 1:] = survival[:, :-1]

        exp_interest = s_start * interest
        exp_principal = s_start * sched_principal
        exp_prepay = f_prepay * upb
        exp_recovery = f_default * upb * (1.0 - cfg.lgd)
        exp_loss = f_default * upb * cfg.lgd
        total_cf = exp_interest + exp_principal + exp_prepay + exp_recovery

        return {
            'interest': exp_interest,
            'scheduled_principal': exp_principal,
            'prepayment': exp_prepay,
            'recovery': exp_recovery,
            'loss': exp_loss,
            'total_cf': total_cf,
            'survival': survival,
            'f_prepay': f_prepay,
            'f_default': f_default,
            'upb_schedule': upb,
        }


def _train_deephit_loop(model, criterion, X_train, y_train, e_train,
                         X_val, y_val, e_val, time_bins,
                         batch_size=256, epochs=100, lr=0.01, patience=10):
    """Train DeepHit with early stopping. Returns training history."""
    device = next(model.parameters()).device
    time_bins_d = time_bins.to(device)

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32)
    e_train_t = torch.tensor(e_train, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).to(device)
    e_val_t = torch.tensor(e_val, dtype=torch.float32).to(device)

    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_train_t, y_train_t, e_train_t),
        batch_size=batch_size, shuffle=True,
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5,
    )

    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        for bx, by, be in loader:
            bx, by, be = bx.to(device), by.to(device), be.to(device)
            optimizer.zero_grad()
            loss, _, _ = criterion(model(bx), by, be, time_bins_d)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss, _, _ = criterion(model(X_val_t), y_val_t, e_val_t, time_bins_d)
            val_loss = val_loss.item()
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 20 == 0:
            print(f'    Epoch {epoch:>3d}  val_loss={val_loss:.4f}')
        if epochs_no_improve >= patience:
            print(f'    Early stopping at epoch {epoch} (best val_loss={best_val_loss:.4f})')
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_val_loss


# === DeepHit hyperparameters ===
DEEPHIT_PARAMS = {
    'num_durations': 200,
    'batch_size': 256,
    'epochs': 100,
    'learning_rate': 0.01,
    'alpha': 1.0,       # beta: weight for ranking loss
    'sigma': 0.1,
    'dropout': 0.6,
    'batch_norm': True,
}


def train_deephit_models(panel_df, cutoff, feature_cols, params):
    """
    Train DeepHit on pre-cutoff terminal observations (no data leakage).

    Returns (dh_engine, feature_cols_used, n_train, n_prepay, n_default, domain_max)
    """
    from sklearn.preprocessing import StandardScaler

    # --- Prepare terminal observations (same logic as RSF) ---
    before = panel_df[panel_df['year_month'] <= cutoff].copy()
    before = before.sort_values(['loan_sequence_number', 'loan_age'])
    terminal_df = before.groupby('loan_sequence_number').last().reset_index()

    # Lag bal_repaid
    def get_lagged_bal_repaid(group):
        return group['bal_repaid'].iloc[-2] if len(group) >= 2 else group['bal_repaid'].iloc[-1]
    bal_repaid_lag = before.groupby('loan_sequence_number').apply(get_lagged_bal_repaid)
    terminal_df['bal_repaid_lag1'] = terminal_df['loan_sequence_number'].map(bal_repaid_lag)

    # Log transform UPB
    terminal_df['log_upb'] = np.log(terminal_df['orig_upb'].astype(float))

    terminal_df = terminal_df.dropna(subset=feature_cols).copy()

    duration = terminal_df['loan_age'].values.astype(np.float32)
    event_code = terminal_df['event_code'].values.astype(np.float32)
    n_loans = len(terminal_df)
    n_prepay = int((event_code == 1).sum())
    n_default = int((event_code == 2).sum())

    # --- Train/val split (80/20 random) ---
    np.random.seed(42)
    idx = np.random.permutation(n_loans)
    split = int(0.8 * n_loans)
    train_idx, val_idx = idx[:split], idx[split:]

    # --- Standardize features ---
    scaler = StandardScaler()
    X_all = terminal_df[feature_cols].values.astype(np.float32)
    X_train = scaler.fit_transform(X_all[train_idx])
    X_val = scaler.transform(X_all[val_idx])

    y_train, y_val = duration[train_idx], duration[val_idx]
    e_train, e_val = event_code[train_idx], event_code[val_idx]

    # --- Time bins ---
    all_dur = duration
    time_bins = np.linspace(all_dur.min(), all_dur.max(), params['num_durations'] + 1)
    time_bins_tensor = torch.tensor(time_bins, dtype=torch.float32)
    time_points = (time_bins[:-1] + time_bins[1:]) / 2  # bin centers

    # --- Build and train model ---
    in_features = X_train.shape[1]
    model = DeepHitNetwork(
        in_features=in_features,
        num_time_bins=params['num_durations'],
        num_causes=2,
        dropout=params['dropout'],
        batch_norm=params['batch_norm'],
    )

    criterion = DeepHitLoss(alpha=params['alpha'], sigma=params['sigma'])

    _train_deephit_loop(
        model, criterion,
        X_train, y_train, e_train,
        X_val, y_val, e_val,
        time_bins_tensor,
        batch_size=params['batch_size'],
        epochs=params['epochs'],
        lr=params['learning_rate'],
        patience=10,
    )
    model.eval()

    domain_max = int(time_points[-1])

    engine = JointDeepHitCashFlowEngine(
        joint_model=model,
        scaler=scaler,
        time_points=time_points,
        config=DeepHitCashFlowConfig(lgd=LGD, projection_horizon=MAX_HORIZON),
    )

    return engine, feature_cols, n_loans, n_prepay, n_default, domain_max

In [ ]:
def build_frozen_cox_matrix(active_loans, feature_names, horizon):
    """
    Build 3D covariate matrix (N, T, F) for Cox engine.

    - Deterministic: bal_repaid (amortization), t_act_12m (min(12, age))
    - Frozen at cutoff: all macro features, delinquency counts
    - Static: int_rate, orig_upb, fico_score, dti_r, ltv_r
    """
    N = len(active_loans)
    F = len(feature_names)
    feat_idx = {name: i for i, name in enumerate(feature_names)}

    X = np.zeros((N, horizon, F), dtype=np.float32)

    # Tile cutoff-row feature values across all months (frozen baseline)
    cutoff_features = active_loans[feature_names].values.astype(np.float32)
    for t in range(horizon):
        X[:, t, :] = cutoff_features

    # Overwrite deterministic features that evolve
    int_rate = active_loans['int_rate'].values.astype(np.float32)
    orig_upb = active_loans['orig_upb'].values.astype(np.float32)
    term = active_loans['orig_loan_term'].values.astype(np.float32)
    current_age = active_loans['current_loan_age'].values.astype(np.float32)

    # Amortization: compute bal_repaid at each future month
    monthly_rate = int_rate / 100.0 / 12.0
    payment = np.where(
        monthly_rate > 0,
        orig_upb * monthly_rate / (1.0 - (1.0 + monthly_rate) ** (-term)),
        orig_upb / term,
    )

    if 'bal_repaid' in feat_idx:
        for t in range(horizon):
            n_payments = current_age + t + 1
            factor = (1.0 + monthly_rate) ** n_payments
            upb_t = np.where(
                monthly_rate > 0,
                orig_upb * factor - payment * (factor - 1.0) / np.where(monthly_rate > 0, monthly_rate, 1.0),
                orig_upb - payment * n_payments,
            )
            upb_t = np.maximum(upb_t, 0.0)
            bal_repaid_t = (orig_upb - upb_t) / orig_upb * 100.0
            X[:, t, feat_idx['bal_repaid']] = bal_repaid_t

    # t_act_12m: deterministic, assuming performing
    if 't_act_12m' in feat_idx:
        for t in range(horizon):
            loan_age = current_age + t + 1
            X[:, t, feat_idx['t_act_12m']] = np.minimum(12.0, loan_age)

    return X

In [ ]:
def build_rsf_snapshot(active_loans, rsf_feature_cols):
    """
    Build 2D feature matrix (N, F) for RSF from cutoff-date snapshot.
    """
    return active_loans[rsf_feature_cols].values.astype(np.float64)

In [ ]:
def compute_realized(panel_df, active_loan_ids, cutoff, horizon):
    """
    Compute realized CIF for active loans after the cutoff using the
    Aalen-Johansen estimator (proper competing-risk CIF with censoring).

    For each active loan, we observe:
    - duration: months from cutoff to event or right-censoring (end of data)
    - event_code: 1 (prepay), 2 (default), 0 (censored)

    Returns dict with:
      - cif_prepay, cif_default: arrays of shape (horizon,)
      - n_prepay, n_default: event counts within horizon
      - n_active: total active loans
    """
    active_loan_ids = set(active_loan_ids)
    N = len(active_loan_ids)

    # Get all post-cutoff rows for active loans
    future = panel_df[
        panel_df['loan_sequence_number'].isin(active_loan_ids) &
        (panel_df['year_month'] > cutoff)
    ].copy()

    # For each loan: last observation after cutoff gives duration and event
    loan_last = future.groupby('loan_sequence_number').last().reset_index()

    # Duration = months from cutoff to last observation
    loan_last['duration'] = (loan_last['year_month'] - cutoff).apply(lambda x: x.n)

    # Event code: 0 if censored (event==0 on last row), else event_code
    duration = loan_last['duration'].values.astype(float)
    event_code = np.where(
        loan_last['event'] == 1,
        loan_last['event_code'].values,
        0,
    ).astype(int)

    # Loans with no post-cutoff rows: treat as censored at month 0
    # (should be rare — they were active at cutoff)
    seen_ids = set(loan_last['loan_sequence_number'].values)
    missing_ids = active_loan_ids - seen_ids
    if missing_ids:
        extra_dur = np.zeros(len(missing_ids))
        extra_evt = np.zeros(len(missing_ids), dtype=int)
        duration = np.concatenate([duration, extra_dur])
        event_code = np.concatenate([event_code, extra_evt])

    # Fit Aalen-Johansen for prepay and default
    ajf_prepay = AalenJohansenFitter(calculate_variance=False)
    ajf_prepay.fit(duration, event_code, event_of_interest=1)

    ajf_default = AalenJohansenFitter(calculate_variance=False)
    ajf_default.fit(duration, event_code, event_of_interest=2)

    # Evaluate CIF on monthly grid 1..horizon
    t_grid = np.arange(1, horizon + 1, dtype=float)

    cif_p = ajf_prepay.cumulative_density_
    cif_d = ajf_default.cumulative_density_

    # Reindex to monthly grid (forward-fill for months with no events)
    cif_prepay = cif_p.reindex(t_grid).ffill().fillna(0.0).iloc[:, 0].values
    cif_default = cif_d.reindex(t_grid).ffill().fillna(0.0).iloc[:, 0].values

    # Count events within horizon
    n_prepay = int(((event_code == 1) & (duration <= horizon)).sum())
    n_default = int(((event_code == 2) & (duration <= horizon)).sum())

    return {
        'cif_prepay': cif_prepay,
        'cif_default': cif_default,
        'n_prepay': n_prepay,
        'n_default': n_default,
        'n_active': N,
    }

In [ ]:
def compute_realized_cash_flows(panel_df, active_loans, cutoff, horizon, lgd=0.25):
    """
    Reconstruct realized portfolio cash flows from post-cutoff panel data.

    Uses observed UPB (derived from bal_repaid) instead of theoretical
    amortization, matching the methodology in notebook 17.

    - Active-at-start loans pay interest + scheduled principal
    - Prepaying loans additionally return remaining UPB
    - Defaulting loans additionally generate recovery = UPB × (1-LGD)

    Returns realized_cf array of shape (N, T).
    """
    N = len(active_loans)
    T = horizon

    loan_ids = active_loans['loan_sequence_number'].values
    loan_idx = {lid: i for i, lid in enumerate(loan_ids)}
    int_rate = active_loans['int_rate'].values.astype(np.float64)
    orig_upb = active_loans['orig_upb'].values.astype(np.float64)
    monthly_rate = int_rate / 100.0 / 12.0

    # --- Get post-cutoff panel rows for these loans ---
    future = panel_df[
        panel_df['loan_sequence_number'].isin(set(loan_ids)) &
        (panel_df['year_month'] > cutoff)
    ].sort_values(['loan_sequence_number', 'year_month']).copy()

    # Recover observed UPB from bal_repaid (matching notebook 17)
    future['observed_upb'] = (
        future['orig_upb'] * (1.0 - future['bal_repaid'].fillna(0.0) / 100.0)
    ).clip(lower=0.0)

    # --- Build observed UPB grid (N, T) ---
    # upb_after[i, t] = observed UPB at month t+1 after cutoff
    upb_after = np.full((N, T), np.nan, dtype=np.float64)

    for lid, group in future.groupby('loan_sequence_number'):
        if lid not in loan_idx:
            continue
        i = loan_idx[lid]
        for _, row in group.iterrows():
            m = int((row['year_month'] - cutoff).n) - 1  # 0-indexed
            if 0 <= m < T:
                upb_after[i, m] = row['observed_upb']

    # upb_start[i, t]: UPB at start of month t (before payment)
    # For t=0, use the cutoff-date observed UPB
    cutoff_upb = np.where(
        active_loans['bal_repaid'].notna().values,
        orig_upb * (1.0 - active_loans['bal_repaid'].fillna(0.0).values / 100.0),
        orig_upb,
    ).clip(min=0.0)

    upb_start = np.full((N, T), np.nan, dtype=np.float64)
    upb_start[:, 0] = cutoff_upb
    for t in range(1, T):
        upb_start[:, t] = np.where(np.isnan(upb_after[:, t - 1]), upb_start[:, t - 1], upb_after[:, t - 1])

    # Fill NaN values: for months beyond data, set both to previous upb_start
    # (loan is still active but unobserved — no cash flow generated)
    for t in range(T):
        mask = np.isnan(upb_after[:, t])
        upb_after[mask, t] = upb_start[mask, t]  # no principal change if unobserved
        mask2 = np.isnan(upb_start[:, t])
        upb_start[mask2, t] = 0.0

    # --- Interest and principal ---
    interest = upb_start * monthly_rate[:, None]
    principal = np.maximum(upb_start - upb_after, 0.0)

    # --- Find realized events per loan ---
    events = future[future['event'] == 1].groupby('loan_sequence_number').first()

    event_month = np.full(N, -1, dtype=int)
    event_type = np.zeros(N, dtype=int)

    for lid, row in events.iterrows():
        if lid not in loan_idx:
            continue
        i = loan_idx[lid]
        m = int((row['year_month'] - cutoff).n) - 1  # 0-indexed
        if 0 <= m < T:
            event_month[i] = m
            event_type[i] = int(row['event_code'])

    # --- Build realized cash flows ---
    # Active indicator: 1 if loan is active at start of month t
    active_indicator = np.ones((N, T), dtype=np.float64)
    for i in range(N):
        if event_month[i] >= 0 and event_month[i] + 1 < T:
            active_indicator[i, event_month[i] + 1:] = 0.0

    # Base: interest + scheduled principal while active (exclude event months for prepay/default)
    is_prepay_month = np.zeros((N, T), dtype=bool)
    is_default_month = np.zeros((N, T), dtype=bool)
    for i in range(N):
        if event_month[i] >= 0:
            if event_type[i] == 1:
                is_prepay_month[i, event_month[i]] = True
            elif event_type[i] == 2:
                is_default_month[i, event_month[i]] = True

    realized_cf = active_indicator * interest
    # Add principal only for non-event months
    realized_cf += active_indicator * principal * (~is_prepay_month & ~is_default_month)

    # Prepayment: return UPB at start of event month
    for i in np.where((event_type == 1) & (event_month >= 0))[0]:
        realized_cf[i, event_month[i]] += upb_start[i, event_month[i]]

    # Default: recovery on UPB at start of event month
    for i in np.where((event_type == 2) & (event_month >= 0))[0]:
        realized_cf[i, event_month[i]] += upb_start[i, event_month[i]] * (1.0 - lgd)

    return realized_cf


def compute_pv(cf_array, annual_rate, axis=None):
    """
    Compute present value of cash flows.

    Parameters
    ----------
    cf_array : np.ndarray
        Cash flows. If 2D (N, T), discounts along axis=1 by default.
    annual_rate : float
        Annual discount rate.
    axis : int, optional
        Axis to sum over. If None, sums all.
    """
    T = cf_array.shape[-1]
    monthly_rate = annual_rate / 12.0
    discount_factors = 1.0 / (1.0 + monthly_rate) ** np.arange(1, T + 1)

    if cf_array.ndim == 2:
        pv = (cf_array * discount_factors[None, :]).sum(axis=1 if axis is None else axis)
    else:
        pv = (cf_array * discount_factors).sum()
    return pv

In [ ]:
def fit_conditional_aj(panel_df, active_loans, cutoff, horizon):
    """
    Fit Aalen-Johansen on loans active at cutoff and compute conditional
    forward-looking CIF.

    CIF_forward(t | survived to age a) = [CIF(a+t) - CIF(a)] / S(a)

    Returns dict with cif_prepay, cif_default (arrays of shape (horizon,))
    and max_observed_age (int).
    """
    # Build training data: all loans with observations up to cutoff
    before = panel_df[panel_df['year_month'] <= cutoff]
    loan_last = before.groupby('loan_sequence_number').last().reset_index()

    duration = loan_last['loan_age'].values.astype(float)
    event_code = np.where(
        loan_last['event'] == 1,
        loan_last['event_code'].values,
        0,
    )

    # Fit AJ for prepay and default
    ajf_prepay = AalenJohansenFitter(calculate_variance=False)
    ajf_prepay.fit(duration, event_code, event_of_interest=1)

    ajf_default = AalenJohansenFitter(calculate_variance=False)
    ajf_default.fit(duration, event_code, event_of_interest=2)

    # Get CIF and survival at all time points
    cif_p = ajf_prepay.cumulative_density_.iloc[:, 0]
    cif_d = ajf_default.cumulative_density_.iloc[:, 0]

    # Overall survival: S(t) = 1 - CIF_prepay(t) - CIF_default(t)
    max_t = int(max(cif_p.index.max(), cif_d.index.max())) + 1
    t_grid = np.arange(max_t + 1)
    cif_p_full = cif_p.reindex(t_grid).ffill().fillna(0.0).values
    cif_d_full = cif_d.reindex(t_grid).ffill().fillna(0.0).values
    surv_full = 1.0 - cif_p_full - cif_d_full

    # Conditional forward CIF: average across active loans
    ages = active_loans['current_loan_age'].values.astype(int)
    N = len(ages)

    fwd_cif_p = np.zeros((N, horizon))
    fwd_cif_d = np.zeros((N, horizon))

    for i, a in enumerate(ages):
        a = min(a, max_t)
        s_a = max(surv_full[a], 1e-10)
        for t in range(horizon):
            at = min(a + t + 1, max_t)
            fwd_cif_p[i, t] = max(0, (cif_p_full[at] - cif_p_full[a]) / s_a)
            fwd_cif_d[i, t] = max(0, (cif_d_full[at] - cif_d_full[a]) / s_a)

    return {
        'cif_prepay': fwd_cif_p.mean(axis=0),
        'cif_default': fwd_cif_d.mean(axis=0),
        'max_observed_age': max_t,
    }

---

## Main Computation: Loop Over Cutoff Dates

For each cutoff date:
1. **Retrain** Cox and RSF models on pre-cutoff data only (no data leakage)
2. Identify active loans and extract cutoff-date features
3. Fit nonparametric CIF (Aalen-Johansen) on pre-cutoff data
4. Project Cox model forward with frozen macro
5. Project RSF model forward with snapshot features
6. Extract realized outcomes from post-cutoff panel

In [ ]:
all_results = {}

for cutoff in CUTOFF_DATES:
    cutoff_str = str(cutoff)
    print(f'\n{"=" * 60}')
    print(f'CUTOFF: {cutoff_str}')
    print(f'{"=" * 60}')

    # 1. Active loans at cutoff
    active = get_active_loans_at_cutoff(panel_df, orig_info, cutoff)
    active = active.dropna(subset=COX_FEATURE_NAMES).copy()
    n_active = len(active)
    portfolio_upb = active['orig_upb'].astype(float).sum()
    remaining_term = (active['orig_loan_term'] - active['current_loan_age']).values
    max_remaining = int(remaining_term.max())
    mean_remaining = remaining_term.mean()
    print(f'Active loans: {n_active:,}')
    print(f'Mean loan age: {active["current_loan_age"].mean():.1f} months')
    print(f'Remaining maturity: mean {mean_remaining:.0f} mo, max {max_remaining} mo')
    print(f'Portfolio UPB at cutoff: ${portfolio_upb/1e6:,.1f}M')

    # 2. Train Cox models on pre-cutoff data
    print('\nTraining Cox models on pre-cutoff data...')
    cox_engine, cox_features, n_cox_loans, n_cox_months, n_cox_prepay, n_cox_default, cox_max_obs = \
        train_cox_models(panel_df, cutoff, COX_FEATURE_NAMES, COX_PENALIZER)
    print(f'  Training data: {n_cox_loans:,} loans, {n_cox_months:,} loan-months')
    print(f'  Events: {n_cox_prepay:,} prepays, {n_cox_default:,} defaults')
    print(f'  Baseline hazard observed up to month {cox_max_obs}')
    if cox_max_obs < MAX_HORIZON:
        print(f'  ** CAPPED: baseline hazard flat-extrapolated beyond month {cox_max_obs} **')

    # 3. Train RSF models on pre-cutoff data
    print('Training RSF models on pre-cutoff data...')
    rsf_engine, n_rsf_loans, n_rsf_prepay, n_rsf_default, rsf_domain_max = \
        train_rsf_models(panel_df, cutoff, RSF_FEATURE_COLS, RSF_PARAMS)
    print(f'  Training data: {n_rsf_loans:,} loans')
    print(f'  Events: {n_rsf_prepay:,} prepays, {n_rsf_default:,} defaults')
    print(f'  Survival curve domain: [0, {rsf_domain_max}] months')
    if rsf_domain_max < MAX_HORIZON:
        print(f'  ** CAPPED: hazards set to 0 beyond month {rsf_domain_max} '
              f'(survival frozen, no further events predicted) **')

    # 4. Train DeepHit models on pre-cutoff data
    print('Training DeepHit models on pre-cutoff data...')
    dh_engine, dh_feature_cols, n_dh_loans, n_dh_prepay, n_dh_default, dh_domain_max = \
        train_deephit_models(panel_df, cutoff, DEEPHIT_FEATURE_COLS, DEEPHIT_PARAMS)
    print(f'  Training data: {n_dh_loans:,} loans')
    print(f'  Events: {n_dh_prepay:,} prepays, {n_dh_default:,} defaults')
    print(f'  Domain: [0, {dh_domain_max}] months')
    if dh_domain_max < MAX_HORIZON:
        print(f'  ** CAPPED: hazards set to 0 beyond month {dh_domain_max} '
              f'(survival frozen, no further events predicted) **')

    # 5. Nonparametric CIF (Aalen-Johansen on pre-cutoff data)
    print('Fitting Aalen-Johansen...')
    aj_result = fit_conditional_aj(panel_df, active, cutoff, MAX_HORIZON)
    aj_max_obs = aj_result['max_observed_age']
    print(f'  Max observed loan age in training: {aj_max_obs} months')
    if aj_max_obs < MAX_HORIZON:
        print(f'  ** CAPPED: CIF frozen beyond age {aj_max_obs} '
              f'(no additional events estimated) **')

    # 6. Cox forward projection
    print('Cox forward projection...')
    X_cox = build_frozen_cox_matrix(active, cox_features, MAX_HORIZON)
    cox_cf = cox_engine.project_cash_flows(active, X_cox)
    cox_cif_prepay = np.cumsum(cox_cf['f_prepay'], axis=1).mean(axis=0)
    cox_cif_default = np.cumsum(cox_cf['f_default'], axis=1).mean(axis=0)

    # 7. RSF forward projection
    print('RSF forward projection...')
    rsf_avail = [c for c in RSF_FEATURE_COLS if c in active.columns]
    X_rsf = build_rsf_snapshot(active, rsf_avail)
    rsf_cf = rsf_engine.project_cash_flows(active, X_rsf)
    rsf_cif_prepay = np.cumsum(rsf_cf['f_prepay'], axis=1).mean(axis=0)
    rsf_cif_default = np.cumsum(rsf_cf['f_default'], axis=1).mean(axis=0)

    # 8. DeepHit forward projection
    print('DeepHit forward projection...')
    dh_avail = [c for c in DEEPHIT_FEATURE_COLS if c in active.columns]
    X_dh = build_rsf_snapshot(active, dh_avail)  # same snapshot logic
    dh_cf = dh_engine.project_cash_flows(active, X_dh)
    dh_cif_prepay = np.cumsum(dh_cf['f_prepay'], axis=1).mean(axis=0)
    dh_cif_default = np.cumsum(dh_cf['f_default'], axis=1).mean(axis=0)

    # 9. Realized outcomes (CIF)
    print('Computing realized outcomes...')
    active_ids = set(active['loan_sequence_number'].values)
    realized = compute_realized(panel_df, active_ids, cutoff, MAX_HORIZON)
    print(f'  Realized: {realized["n_prepay"]:,} prepays, '
          f'{realized["n_default"]:,} defaults out of {realized["n_active"]:,} loans')

    # Print CIF at all horizons
    valid_horizons = [h for h in HORIZONS if h <= MAX_HORIZON]
    print(f'\n  {"Horizon":>7s}  {"Realized":>10s}  {"AJ":>10s}  {"Cox":>10s}  {"RSF":>10s}  {"DeepHit":>10s}  (Prepay CIF)')
    for h in valid_horizons:
        t = h - 1
        flags = []
        if h > aj_max_obs:
            flags.append('AJ capped')
        if h > rsf_domain_max:
            flags.append('RSF capped')
        if h > cox_max_obs:
            flags.append('Cox extrapolated')
        if h > dh_domain_max:
            flags.append('DH capped')
        flag_str = f'  [{", ".join(flags)}]' if flags else ''
        print(f'  {h:>5d}m   {realized["cif_prepay"][t]:>10.4f}  '
              f'{aj_result["cif_prepay"][t]:>10.4f}  '
              f'{cox_cif_prepay[t]:>10.4f}  '
              f'{rsf_cif_prepay[t]:>10.4f}  '
              f'{dh_cif_prepay[t]:>10.4f}{flag_str}')

    print(f'\n  {"Horizon":>7s}  {"Realized":>10s}  {"AJ":>10s}  {"Cox":>10s}  {"RSF":>10s}  {"DeepHit":>10s}  (Default CIF)')
    for h in valid_horizons:
        t = h - 1
        flags = []
        if h > aj_max_obs:
            flags.append('AJ capped')
        if h > rsf_domain_max:
            flags.append('RSF capped')
        if h > cox_max_obs:
            flags.append('Cox extrapolated')
        if h > dh_domain_max:
            flags.append('DH capped')
        flag_str = f'  [{", ".join(flags)}]' if flags else ''
        print(f'  {h:>5d}m   {realized["cif_default"][t]:>10.4f}  '
              f'{aj_result["cif_default"][t]:>10.4f}  '
              f'{cox_cif_default[t]:>10.4f}  '
              f'{rsf_cif_default[t]:>10.4f}  '
              f'{dh_cif_default[t]:>10.4f}{flag_str}')

    # 10. Realized cash flows and PV metrics
    print('\nComputing realized cash flows and PV errors...')
    realized_cf = compute_realized_cash_flows(panel_df, active, cutoff, MAX_HORIZON, lgd=LGD)

    # Truncate to observation window
    T_obs = int((panel_df['year_month'].max() - cutoff).n)
    T_obs = min(T_obs, MAX_HORIZON)
    print(f'  Observation window: {T_obs} months (data ends {panel_df["year_month"].max()})')

    # PV of portfolio cash flows (truncated to observation window)
    pv_realized = compute_pv(realized_cf.sum(axis=0)[:T_obs], DISCOUNT_RATE)
    pv_cox = compute_pv(cox_cf['total_cf'].sum(axis=0)[:T_obs], DISCOUNT_RATE)
    pv_rsf = compute_pv(rsf_cf['total_cf'].sum(axis=0)[:T_obs], DISCOUNT_RATE)
    pv_dh = compute_pv(dh_cf['total_cf'].sum(axis=0)[:T_obs], DISCOUNT_RATE)

    pv_err_cox = pv_cox - pv_realized
    pv_err_rsf = pv_rsf - pv_realized
    pv_err_dh = pv_dh - pv_realized
    pv_err_cox_pct = pv_err_cox / portfolio_upb * 100
    pv_err_rsf_pct = pv_err_rsf / portfolio_upb * 100
    pv_err_dh_pct = pv_err_dh / portfolio_upb * 100

    print(f'  PV realized:  ${pv_realized/1e6:,.2f}M')
    print(f'  PV Cox:       ${pv_cox/1e6:,.2f}M  (error: ${pv_err_cox/1e6:,.2f}M = {pv_err_cox_pct:+.2f}% of portfolio)')
    print(f'  PV RSF:       ${pv_rsf/1e6:,.2f}M  (error: ${pv_err_rsf/1e6:,.2f}M = {pv_err_rsf_pct:+.2f}% of portfolio)')
    print(f'  PV DeepHit:   ${pv_dh/1e6:,.2f}M  (error: ${pv_err_dh/1e6:,.2f}M = {pv_err_dh_pct:+.2f}% of portfolio)')

    # Store results
    all_results[cutoff_str] = {
        'n_active': n_active,
        'cutoff': cutoff,
        'portfolio_upb': portfolio_upb,
        'max_remaining': max_remaining,
        'n_cox_train': n_cox_loans,
        'n_rsf_train': n_rsf_loans,
        'n_dh_train': n_dh_loans,
        'cox_max_obs': cox_max_obs,
        'rsf_domain_max': rsf_domain_max,
        'dh_domain_max': dh_domain_max,
        'aj_max_obs': aj_max_obs,
        'aj': aj_result,
        'cox_cif_prepay': cox_cif_prepay,
        'cox_cif_default': cox_cif_default,
        'cox_cf': cox_cf,
        'rsf_cif_prepay': rsf_cif_prepay,
        'rsf_cif_default': rsf_cif_default,
        'rsf_cf': rsf_cf,
        'dh_cif_prepay': dh_cif_prepay,
        'dh_cif_default': dh_cif_default,
        'dh_cf': dh_cf,
        'realized': realized,
        'realized_cf': realized_cf,
        'pv_realized': pv_realized,
        'pv_cox': pv_cox,
        'pv_rsf': pv_rsf,
        'pv_dh': pv_dh,
        'pv_err_cox': pv_err_cox,
        'pv_err_rsf': pv_err_rsf,
        'pv_err_dh': pv_err_dh,
        'pv_err_cox_pct': pv_err_cox_pct,
        'pv_err_rsf_pct': pv_err_rsf_pct,
        'pv_err_dh_pct': pv_err_dh_pct,
        'T_obs': T_obs,
    }

print(f'\n{"=" * 60}')
print('All cutoffs complete.')

# Summary of model domain limits
print(f'\n--- Model Domain Limits ---')
for cutoff_str, res in all_results.items():
    print(f'{cutoff_str}: Cox h0 observed to {res["cox_max_obs"]}m '
          f'(flat-extrapolated to 360), '
          f'RSF domain {res["rsf_domain_max"]}m '
          f'(hazards=0 beyond), '
          f'DeepHit domain {res["dh_domain_max"]}m '
          f'(hazards=0 beyond), '
          f'AJ max age {res["aj_max_obs"]}m '
          f'(CIF frozen beyond), '
          f'max remaining maturity {res["max_remaining"]}m')

---

## CIF Comparison: Predicted vs Realized

In [ ]:
# Build comparison table: CIF at each horizon for each cutoff and model
rows = []
for cutoff_str, res in all_results.items():
    real = res['realized']
    for h in HORIZONS:
        T_obs = res['T_obs']
        if h > T_obs:
            continue
        t = h - 1  # 0-indexed
        rows.append({
            'cutoff': cutoff_str,
            'horizon': h,
            'real_cif_p': real['cif_prepay'][t],
            'real_cif_d': real['cif_default'][t],
            'aj_cif_p': res['aj']['cif_prepay'][t],
            'aj_cif_d': res['aj']['cif_default'][t],
            'cox_cif_p': res['cox_cif_prepay'][t],
            'cox_cif_d': res['cox_cif_default'][t],
            'rsf_cif_p': res['rsf_cif_prepay'][t],
            'rsf_cif_d': res['rsf_cif_default'][t],
            'dh_cif_p': res['dh_cif_prepay'][t],
            'dh_cif_d': res['dh_cif_default'][t],
        })

cif_df = pd.DataFrame(rows)

# Compute errors
for model in ['aj', 'cox', 'rsf', 'dh']:
    cif_df[f'{model}_err_p'] = cif_df[f'{model}_cif_p'] - cif_df['real_cif_p']
    cif_df[f'{model}_err_d'] = cif_df[f'{model}_cif_d'] - cif_df['real_cif_d']

# Print
print('=== CIF Comparison: Predicted vs Realized ===')
print()
for event, suffix in [('Prepayment', '_p'), ('Default', '_d')]:
    print(f'--- {event} CIF ---')
    cols = ['cutoff', 'horizon', f'real_cif{suffix}',
            f'aj_cif{suffix}', f'aj_err{suffix}',
            f'cox_cif{suffix}', f'cox_err{suffix}',
            f'rsf_cif{suffix}', f'rsf_err{suffix}',
            f'dh_cif{suffix}', f'dh_err{suffix}']
    print(cif_df[cols].to_string(index=False, float_format='{:.4f}'.format))
    print()

---

## CIF Curves: Predicted vs Realized

In [ ]:
# Prepayment CIF curves
n_cutoffs = len(CUTOFF_DATES)
fig, axes = plt.subplots(1, n_cutoffs, figsize=(max(6, 4 * n_cutoffs), 4), sharey=True, squeeze=False)

for i, (cutoff_str, res) in enumerate(all_results.items()):
    ax = axes[0, i]
    real = res['realized']
    T = res['T_obs']
    months = np.arange(1, T + 1)

    ax.plot(months, real['cif_prepay'][:T], 'k-', lw=2, label='Realized (AJ)')
    ax.plot(months, res['aj']['cif_prepay'][:T], '--', color='green', lw=1.5, label='AJ (pre-cutoff)')
    ax.plot(months, res['cox_cif_prepay'][:T], '--', color='steelblue', lw=1.5, label='Cox')
    ax.plot(months, res['rsf_cif_prepay'][:T], '--', color='darkorange', lw=1.5, label='RSF')
    ax.plot(months, res['dh_cif_prepay'][:T], '--', color='crimson', lw=1.5, label='DeepHit')

    ax.set_title(f'Cutoff: {cutoff_str}')
    ax.set_xlabel('Months since cutoff')
    if i == 0:
        ax.set_ylabel('Cumulative Incidence (Prepay)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Prepayment CIF: Predicted vs Realized', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'backtest_cif_prepay.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Default CIF curves
fig, axes = plt.subplots(1, n_cutoffs, figsize=(max(6, 4 * n_cutoffs), 4), sharey=True, squeeze=False)

for i, (cutoff_str, res) in enumerate(all_results.items()):
    ax = axes[0, i]
    real = res['realized']
    T = res['T_obs']
    months = np.arange(1, T + 1)

    ax.plot(months, real['cif_default'][:T], 'k-', lw=2, label='Realized (AJ)')
    ax.plot(months, res['aj']['cif_default'][:T], '--', color='green', lw=1.5, label='AJ (pre-cutoff)')
    ax.plot(months, res['cox_cif_default'][:T], '--', color='steelblue', lw=1.5, label='Cox')
    ax.plot(months, res['rsf_cif_default'][:T], '--', color='darkorange', lw=1.5, label='RSF')
    ax.plot(months, res['dh_cif_default'][:T], '--', color='crimson', lw=1.5, label='DeepHit')

    ax.set_title(f'Cutoff: {cutoff_str}')
    ax.set_xlabel('Months since cutoff')
    if i == 0:
        ax.set_ylabel('Cumulative Incidence (Default)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Default CIF: Predicted vs Realized', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'backtest_cif_default.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Cash Flow Comparison

In [ ]:
# Cumulative total cash flow: predicted (Cox, RSF, DeepHit) vs realized
fig, axes = plt.subplots(1, n_cutoffs, figsize=(max(6, 4 * n_cutoffs), 4), sharey=False, squeeze=False)

for i, (cutoff_str, res) in enumerate(all_results.items()):
    ax = axes[0, i]
    T = res['T_obs']
    m = np.arange(1, T + 1)

    # Portfolio-level cumulative CF ($M)
    cox_cumul = np.cumsum(res['cox_cf']['total_cf'].sum(axis=0))[:T] / 1e6
    rsf_cumul = np.cumsum(res['rsf_cf']['total_cf'].sum(axis=0))[:T] / 1e6
    dh_cumul = np.cumsum(res['dh_cf']['total_cf'].sum(axis=0))[:T] / 1e6
    real_cumul = np.cumsum(res['realized_cf'].sum(axis=0))[:T] / 1e6

    ax.plot(m, real_cumul, 'k-', lw=2, label='Realized')
    ax.plot(m, cox_cumul, '--', color='steelblue', lw=1.5, label='Cox')
    ax.plot(m, rsf_cumul, '--', color='darkorange', lw=1.5, label='RSF')
    ax.plot(m, dh_cumul, '--', color='crimson', lw=1.5, label='DeepHit')

    ax.set_title(f'Cutoff: {cutoff_str}')
    ax.set_xlabel('Months since cutoff')
    if i == 0:
        ax.set_ylabel('Cumulative CF ($M)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Cumulative Cash Flows: Predicted vs Realized', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'backtest_cumul_cf.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Survival Curves: Predicted vs Realized

In [ ]:
# Average survival S(t) = 1 - CIF_prepay(t) - CIF_default(t)
fig, axes = plt.subplots(1, n_cutoffs, figsize=(max(6, 4 * n_cutoffs), 4), sharey=True, squeeze=False)

for i, (cutoff_str, res) in enumerate(all_results.items()):
    ax = axes[0, i]
    real = res['realized']
    T = res['T_obs']
    months = np.arange(1, T + 1)

    real_surv = 1.0 - real['cif_prepay'][:T] - real['cif_default'][:T]
    aj_surv = 1.0 - res['aj']['cif_prepay'][:T] - res['aj']['cif_default'][:T]
    cox_surv = 1.0 - res['cox_cif_prepay'][:T] - res['cox_cif_default'][:T]
    rsf_surv = 1.0 - res['rsf_cif_prepay'][:T] - res['rsf_cif_default'][:T]
    dh_surv = 1.0 - res['dh_cif_prepay'][:T] - res['dh_cif_default'][:T]

    ax.plot(months, real_surv, 'k-', lw=2, label='Realized')
    ax.plot(months, aj_surv, '--', color='green', lw=1.5, label='AJ')
    ax.plot(months, cox_surv, '--', color='steelblue', lw=1.5, label='Cox')
    ax.plot(months, rsf_surv, '--', color='darkorange', lw=1.5, label='RSF')
    ax.plot(months, dh_surv, '--', color='crimson', lw=1.5, label='DeepHit')

    ax.set_title(f'Cutoff: {cutoff_str}')
    ax.set_xlabel('Months since cutoff')
    if i == 0:
        ax.set_ylabel('Survival S(t)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Overall Survival: Predicted vs Realized', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'backtest_survival.png', dpi=150, bbox_inches='tight')
plt.show()

---

## CIF Error Heatmaps

In [ ]:
if len(CUTOFF_DATES) > 1:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for col_idx, model in enumerate(['aj', 'cox', 'rsf']):
        for row_idx, (event, suffix) in enumerate([('Prepayment', '_p'), ('Default', '_d')]):
            ax = axes[row_idx, col_idx]
            pivot = cif_df.pivot(
                index='cutoff', columns='horizon', values=f'{model}_err{suffix}'
            )
            vmax = max(abs(cif_df[f'{model}_err{suffix}']).max(), 0.01)
            sns.heatmap(
                pivot, annot=True, fmt='.4f', cmap='RdBu_r',
                center=0, vmin=-vmax, vmax=vmax, ax=ax,
            )
            ax.set_title(f'{model.upper()} — {event}')
            ax.set_xlabel('Horizon (months)')
            ax.set_ylabel('Cutoff' if col_idx == 0 else '')
else:
    # Single cutoff: bar chart of errors by horizon
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    horizons = cif_df['horizon'].values
    x = np.arange(len(horizons))
    width = 0.25

    for ax, (event, suffix) in zip(axes, [('Prepayment', '_p'), ('Default', '_d')]):
        for j, (model, color) in enumerate([('aj', 'green'), ('cox', 'steelblue'), ('rsf', 'darkorange')]):
            vals = cif_df[f'{model}_err{suffix}'].values
            ax.bar(x + j * width, vals, width, label=model.upper(), color=color, alpha=0.8)
        ax.axhline(0, color='black', lw=0.8)
        ax.set_title(f'{event} CIF Error')
        ax.set_xlabel('Horizon (months)')
        ax.set_ylabel('Predicted - Realized')
        ax.set_xticks(x + width)
        ax.set_xticklabels(horizons)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('CIF Prediction Error (Predicted - Realized)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'backtest_cif_error_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Summary Statistics

In [ ]:
# Mean absolute error across all cutoff-horizon combinations
summary_rows = []
for model in ['aj', 'cox', 'rsf', 'dh']:
    label = model.upper() if model != 'dh' else 'DeepHit'
    mae_p = cif_df[f'{model}_err_p'].abs().mean()
    mae_d = cif_df[f'{model}_err_d'].abs().mean()
    rmse_p = np.sqrt((cif_df[f'{model}_err_p'] ** 2).mean())
    rmse_d = np.sqrt((cif_df[f'{model}_err_d'] ** 2).mean())
    bias_p = cif_df[f'{model}_err_p'].mean()
    bias_d = cif_df[f'{model}_err_d'].mean()
    summary_rows.append({
        'Model': label,
        'MAE Prepay': mae_p,
        'MAE Default': mae_d,
        'RMSE Prepay': rmse_p,
        'RMSE Default': rmse_d,
        'Bias Prepay': bias_p,
        'Bias Default': bias_d,
    })

summary_df = pd.DataFrame(summary_rows)
print('=== Model Comparison: CIF Prediction Accuracy ===')
print(summary_df.to_string(index=False, float_format='{:.5f}'.format))

# Per-cutoff summary including training data size
print('\n=== Per-Cutoff Results ===')
for cutoff_str, res in all_results.items():
    real = res['realized']
    print(f'\n{cutoff_str}:')
    print(f'  Training: Cox={res["n_cox_train"]:,} loans, RSF={res["n_rsf_train"]:,} loans')
    print(f'  Test: {res["n_active"]:,} active loans')
    print(f'  Realized: {real["n_prepay"]:,} prepays ({real["n_prepay"]/res["n_active"]*100:.1f}%), '
          f'{real["n_default"]:,} defaults ({real["n_default"]/res["n_active"]*100:.1f}%)')

---

## Present Value Error: Predicted vs Realized Cash Flows

PV error = PV(predicted) - PV(realized), discounted at the annual rate defined above.
Expressed as a percentage of total portfolio UPB at cutoff.

In [ ]:
# PV error summary table
pv_rows = []
for cutoff_str, res in all_results.items():
    pv_rows.append({
        'Cutoff': cutoff_str,
        'Active loans': f'{res["n_active"]:,}',
        'Portfolio UPB ($M)': f'{res["portfolio_upb"]/1e6:,.1f}',
        'PV Realized ($M)': f'{res["pv_realized"]/1e6:,.2f}',
        'PV Cox ($M)': f'{res["pv_cox"]/1e6:,.2f}',
        'PV RSF ($M)': f'{res["pv_rsf"]/1e6:,.2f}',
        'PV DeepHit ($M)': f'{res["pv_dh"]/1e6:,.2f}',
        'Cox Error ($M)': f'{res["pv_err_cox"]/1e6:+,.2f}',
        'RSF Error ($M)': f'{res["pv_err_rsf"]/1e6:+,.2f}',
        'DH Error ($M)': f'{res["pv_err_dh"]/1e6:+,.2f}',
        'Cox Error (% UPB)': f'{res["pv_err_cox_pct"]:+.2f}%',
        'RSF Error (% UPB)': f'{res["pv_err_rsf_pct"]:+.2f}%',
        'DH Error (% UPB)': f'{res["pv_err_dh_pct"]:+.2f}%',
    })

pv_df = pd.DataFrame(pv_rows)
print(f'=== PV Cash Flow Error (discount rate = {DISCOUNT_RATE:.1%}) ===')
print(pv_df.to_string(index=False))

# Average absolute PV error
avg_cox_pct = np.mean([abs(r['pv_err_cox_pct']) for r in all_results.values()])
avg_rsf_pct = np.mean([abs(r['pv_err_rsf_pct']) for r in all_results.values()])
avg_dh_pct = np.mean([abs(r['pv_err_dh_pct']) for r in all_results.values()])
print(f'\nMean |PV Error| as % of portfolio UPB:')
print(f'  Cox:     {avg_cox_pct:.2f}%')
print(f'  RSF:     {avg_rsf_pct:.2f}%')
print(f'  DeepHit: {avg_dh_pct:.2f}%')

In [ ]:
# PV error bar chart
cutoff_labels = list(all_results.keys())
cox_pv_errs = [all_results[c]['pv_err_cox_pct'] for c in cutoff_labels]
rsf_pv_errs = [all_results[c]['pv_err_rsf_pct'] for c in cutoff_labels]
dh_pv_errs = [all_results[c]['pv_err_dh_pct'] for c in cutoff_labels]

x = np.arange(len(cutoff_labels))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width, cox_pv_errs, width, label='Cox', color='steelblue', alpha=0.8)
bars2 = ax.bar(x, rsf_pv_errs, width, label='RSF', color='darkorange', alpha=0.8)
bars3 = ax.bar(x + width, dh_pv_errs, width, label='DeepHit', color='crimson', alpha=0.8)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Cutoff Date')
ax.set_ylabel('PV Error (% of Portfolio UPB)')
ax.set_title(f'Present Value Cash Flow Error by Cutoff (discount rate = {DISCOUNT_RATE:.1%})')
ax.set_xticks(x)
ax.set_xticklabels(cutoff_labels)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h, f'{h:+.2f}%',
                ha='center', va='bottom' if h >= 0 else 'top', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'backtest_pv_error.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Monthly PV of Cash Flow Differences

For each month $t$ after the cutoff, we compute the **discounted cash flow error**:

$$\text{PV diff}(t) = \frac{\text{CF}_{\text{predicted}}(t) - \text{CF}_{\text{realized}}(t)}{(1 + r/12)^t}$$

The **cumulative PV difference** shows how the pricing error builds over time.
We express the final cumulative value as a percentage of the portfolio
notional (total UPB of active loans at cutoff).

In [ ]:
monthly_disc = DISCOUNT_RATE / 12.0

for cutoff_str, res in all_results.items():
    portfolio_upb = res['portfolio_upb']
    T_obs = res['T_obs']

    realized_monthly = res['realized_cf'].sum(axis=0)[:T_obs]
    cox_monthly = res['cox_cf']['total_cf'].sum(axis=0)[:T_obs]
    rsf_monthly = res['rsf_cf']['total_cf'].sum(axis=0)[:T_obs]
    dh_monthly = res['dh_cf']['total_cf'].sum(axis=0)[:T_obs]

    T = T_obs

    months = np.arange(1, T + 1)
    discount_factors = 1.0 / (1.0 + monthly_disc) ** months

    # Monthly nominal difference
    cox_diff = cox_monthly - realized_monthly
    rsf_diff = rsf_monthly - realized_monthly
    dh_diff = dh_monthly - realized_monthly

    # Monthly discounted difference
    cox_pv_diff = cox_diff * discount_factors
    rsf_pv_diff = rsf_diff * discount_factors
    dh_pv_diff = dh_diff * discount_factors

    # Cumulative PV difference
    cox_cum_pv = np.cumsum(cox_pv_diff)
    rsf_cum_pv = np.cumsum(rsf_pv_diff)
    dh_cum_pv = np.cumsum(dh_pv_diff)

    # --- Plots ---
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # 1. Monthly nominal CF: predicted vs realized
    ax = axes[0, 0]
    ax.plot(months, realized_monthly / 1e6, 'k-', lw=2, label='Realized')
    ax.plot(months, cox_monthly / 1e6, '--', color='steelblue', lw=1.5, label='Cox')
    ax.plot(months, rsf_monthly / 1e6, '--', color='darkorange', lw=1.5, label='RSF')
    ax.plot(months, dh_monthly / 1e6, '--', color='crimson', lw=1.5, label='DeepHit')
    ax.set_title('Monthly Portfolio Cash Flow')
    ax.set_xlabel('Months after cutoff')
    ax.set_ylabel('$M')
    ax.legend(fontsize=9)

    # 2. Monthly discounted difference
    ax = axes[0, 1]
    ax.bar(months, cox_pv_diff / 1e6, width=0.8, color='steelblue', alpha=0.5, label='Cox')
    ax.bar(months, rsf_pv_diff / 1e6, width=0.5, color='darkorange', alpha=0.5, label='RSF')
    ax.bar(months, dh_pv_diff / 1e6, width=0.3, color='crimson', alpha=0.5, label='DeepHit')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title('Monthly PV of Cash Flow Difference')
    ax.set_xlabel('Months after cutoff')
    ax.set_ylabel('$M (discounted)')
    ax.legend(fontsize=9)

    # 3. Cumulative PV difference
    ax = axes[1, 0]
    ax.plot(months, cox_cum_pv / 1e6, '-', color='steelblue', lw=2, label='Cox')
    ax.plot(months, rsf_cum_pv / 1e6, '-', color='darkorange', lw=2, label='RSF')
    ax.plot(months, dh_cum_pv / 1e6, '-', color='crimson', lw=2, label='DeepHit')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title('Cumulative PV Difference (Predicted - Realized)')
    ax.set_xlabel('Months after cutoff')
    ax.set_ylabel('$M (cumulative discounted)')
    ax.legend(fontsize=9)

    # 4. Cumulative PV difference as % of portfolio UPB
    ax = axes[1, 1]
    ax.plot(months, cox_cum_pv / portfolio_upb * 100, '-', color='steelblue', lw=2, label='Cox')
    ax.plot(months, rsf_cum_pv / portfolio_upb * 100, '-', color='darkorange', lw=2, label='RSF')
    ax.plot(months, dh_cum_pv / portfolio_upb * 100, '-', color='crimson', lw=2, label='DeepHit')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title('Cumulative PV Difference as % of Portfolio Notional')
    ax.set_xlabel('Months after cutoff')
    ax.set_ylabel('% of UPB')
    ax.legend(fontsize=9)

    for ax in axes.flat:
        ax.grid(True, alpha=0.3)

    plt.suptitle(f'Cutoff {cutoff_str} — PV of Monthly Cash Flow Differences '
                 f'(r = {DISCOUNT_RATE:.1%})',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'backtest_monthly_pv_diff_{cutoff_str}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # --- Summary table ---
    print(f'\n=== Cutoff {cutoff_str}: Cumulative PV Difference ===')
    print(f'Portfolio notional (UPB at cutoff): ${portfolio_upb/1e6:,.1f}M')
    print(f'Discount rate: {DISCOUNT_RATE:.1%} annual')
    print(f'')
    print(f'{"Horizon":>8s}  {"Cox PV diff ($M)":>18s}  {"Cox % UPB":>10s}  '
          f'{"RSF PV diff ($M)":>18s}  {"RSF % UPB":>10s}  '
          f'{"DH PV diff ($M)":>18s}  {"DH % UPB":>10s}')
    print('-' * 105)
    for h in [6, 12, 24, 36, 48, 60, T]:
        if h > T:
            continue
        t = h - 1
        label = f'{h}m' if h < T else f'{T}m (all)'
        print(f'{label:>8s}  {cox_cum_pv[t]/1e6:>+18.2f}  {cox_cum_pv[t]/portfolio_upb*100:>+10.3f}%  '
              f'{rsf_cum_pv[t]/1e6:>+18.2f}  {rsf_cum_pv[t]/portfolio_upb*100:>+10.3f}%  '
              f'{dh_cum_pv[t]/1e6:>+18.2f}  {dh_cum_pv[t]/portfolio_upb*100:>+10.3f}%')

---

## Conclusions

### Methodology
- **No data leakage**: All models retrained at the cutoff using only pre-cutoff data
- **Forward projection**: Features frozen at cutoff values (except deterministic evolution of `bal_repaid` and `t_act_12m`)
- **Fair comparison**: All models evaluated on the same loan cohort and observation window

### Key findings
- The cutoff date (2021-06) sits in the middle of a **major regime shift**: post-COVID rate
  environment with historically low mortgage rates transitioning to rapid rate hikes
- **Aalen-Johansen**: Purely nonparametric, captures average historical behavior but cannot
  adapt to changing macro conditions. Tends to over-predict prepayment for this cohort
- **Cox TV**: Captures macro sensitivity through time-varying covariates, but freezing
  features at cutoff misses the dramatic rate changes that followed
- **RSF**: Snapshot-based prediction captures nonlinear feature interactions but similarly
  cannot anticipate post-cutoff macro shifts
- **DeepHit**: Neural network approach with joint competing risks modeling. Like RSF,
  uses snapshot features and cannot foresee post-cutoff regime changes
- PV errors reflect the fundamental challenge of forward-looking projection through
  structural breaks in the economic environment

---

## Loan-Level PV Error by Forward Bucket

For each loan in the cutoff cohort, we compute the per-loan cash flow
difference (predicted - realized) in each forward-looking time bucket.
Each bucket's difference is discounted to present value at the bucket
midpoint. Error metrics (ME, MAE, RMSE) are then computed across loans
on these PV differences.

This answers: **how large is the per-loan pricing error, in PV terms,
at different horizons?**

In [ ]:
# --- Configuration ---
# Time buckets: (start_month, end_month) — 0-indexed, end exclusive
BUCKETS = [(0, 3), (3, 6), (6, 12)]
for start in range(12, 48, 6):
    BUCKETS.append((start, start + 6))

BUCKET_LABELS = [f'{b[0]}-{b[1]}m' for b in BUCKETS]
BUCKET_MIDPOINTS = [(b[0] + b[1]) / 2.0 for b in BUCKETS]

MODEL_NAMES = ['AJ', 'Cox', 'RSF', 'DeepHit']
monthly_disc = DISCOUNT_RATE / 12.0

print(f'Buckets: {BUCKET_LABELS}')
print(f'Discount rate: {DISCOUNT_RATE:.1%} annual')

In [ ]:
# Use the cutoff cohort results from the main loop
cutoff_str = str(CUTOFF_DATES[0])
res = all_results[cutoff_str]
T_obs = res['T_obs']
portfolio_upb = res['portfolio_upb']

# Realized CF: shape (N, T) — already computed in main loop
realized_cf = res['realized_cf'][:, :T_obs]
N = realized_cf.shape[0]

# Predicted CF per model
# Cox, RSF, and DeepHit are already in results; AJ needs loan-level computation
cox_pred = res['cox_cf']['total_cf'][:, :T_obs]
rsf_pred = res['rsf_cf']['total_cf'][:, :T_obs]
dh_pred = res['dh_cf']['total_cf'][:, :T_obs]

# AJ loan-level predicted CF
from lifelines import AalenJohansenFitter

cutoff = CUTOFF_DATES[0]
active = get_active_loans_at_cutoff(panel_df, orig_info, cutoff)
active = active.dropna(subset=COX_FEATURE_NAMES).copy()

# Fit AJ on pre-cutoff data
before = panel_df[panel_df['year_month'] <= cutoff]
loan_last = before.groupby('loan_sequence_number').last().reset_index()
duration = loan_last['loan_age'].values.astype(float)
event_code = np.where(
    loan_last['event'] == 1, loan_last['event_code'].values, 0
).astype(int)

ajf_p = AalenJohansenFitter(calculate_variance=False)
ajf_p.fit(duration, event_code, event_of_interest=1)
ajf_d = AalenJohansenFitter(calculate_variance=False)
ajf_d.fit(duration, event_code, event_of_interest=2)

cif_p_raw = ajf_p.cumulative_density_.iloc[:, 0]
cif_d_raw = ajf_d.cumulative_density_.iloc[:, 0]
max_t = int(max(cif_p_raw.index.max(), cif_d_raw.index.max())) + 1
t_grid = np.arange(max_t + 1)
cif_p_full = cif_p_raw.reindex(t_grid).ffill().fillna(0.0).values
cif_d_full = cif_d_raw.reindex(t_grid).ffill().fillna(0.0).values
surv_full = 1.0 - cif_p_full - cif_d_full

# AJ per-loan CF using conditional CIF
int_rate = active['int_rate'].values.astype(np.float64)
orig_upb = active['orig_upb'].values.astype(np.float64)
term = active['orig_loan_term'].values.astype(np.float64)
current_age = active['current_loan_age'].values.astype(np.float64)
monthly_rate = int_rate / 100.0 / 12.0

payment = np.where(
    monthly_rate > 0,
    orig_upb * monthly_rate / (1.0 - (1.0 + monthly_rate) ** (-term)),
    orig_upb / term,
)
factor_start = (1.0 + monthly_rate) ** current_age
upb_cutoff = np.where(
    monthly_rate > 0,
    orig_upb * factor_start - payment * (factor_start - 1.0) / monthly_rate,
    orig_upb - payment * current_age,
)
upb_cutoff = np.maximum(upb_cutoff, 0.0)

aj_pred = np.zeros((N, T_obs), dtype=np.float64)
prev_upb = upb_cutoff.copy()
for t in range(T_obs):
    total_age = current_age + t + 1
    active_mask = (total_age <= term).astype(float)
    int_t = prev_upb * monthly_rate * active_mask
    prin_t = np.maximum(np.minimum(payment - int_t, prev_upb), 0.0) * active_mask
    new_upb = np.maximum(prev_upb - prin_t, 0.0)
    for i in range(N):
        a = min(int(current_age[i]), max_t)
        s_a = max(surv_full[a], 1e-10)
        at = min(a + t + 1, max_t)
        at_prev = min(a + t, max_t)
        s_prev = max((surv_full[at_prev] / s_a) if t > 0 else 1.0, 0.0)
        f_p = max((cif_p_full[at] - cif_p_full[at_prev]) / s_a, 0.0)
        f_d = max((cif_d_full[at] - cif_d_full[at_prev]) / s_a, 0.0)
        aj_pred[i, t] = (
            s_prev * (int_t[i] + prin_t[i])
            + f_p * new_upb[i]
            + f_d * new_upb[i] * (1.0 - LGD)
        )
    prev_upb = new_upb

predictions = {'AJ': aj_pred, 'Cox': cox_pred, 'RSF': rsf_pred, 'DeepHit': dh_pred}

print(f'Cohort: {N:,} loans, {T_obs} months observation window')
print(f'Portfolio UPB at cutoff: ${portfolio_upb/1e6:,.1f}M')

In [ ]:
# --- Compute PV of per-loan errors by bucket ---
pv_errors = {m: {} for m in MODEL_NAMES}  # {model: {bucket: array(N,)}}

for model_name in MODEL_NAMES:
    pred = predictions[model_name]
    error = pred - realized_cf  # (N, T_obs)

    for bucket_label, (bstart, bend) in zip(BUCKET_LABELS, BUCKETS):
        if bstart >= T_obs:
            break
        bend_actual = min(bend, T_obs)
        midpoint = (bstart + bend_actual) / 2.0

        # Discount each month within the bucket individually
        months_in_bucket = np.arange(bstart, bend_actual)
        disc_factors = 1.0 / (1.0 + monthly_disc) ** (months_in_bucket + 1)

        # PV of per-loan error for this bucket
        bucket_pv_err = (error[:, bstart:bend_actual] * disc_factors[None, :]).sum(axis=1)
        pv_errors[model_name][bucket_label] = bucket_pv_err

# --- Build metrics table ---
for model_name in MODEL_NAMES:
    print(f'\n{"=" * 70}')
    print(f'{model_name}: Per-Loan PV Error by Bucket (discount rate = {DISCOUNT_RATE:.1%})')
    print(f'{"=" * 70}')

    metrics_rows = []
    for bucket_label in BUCKET_LABELS:
        if bucket_label not in pv_errors[model_name]:
            break
        pv_err = pv_errors[model_name][bucket_label]

        me = pv_err.mean()
        mae = np.abs(pv_err).mean()
        rmse = np.sqrt((pv_err ** 2).mean())
        total_pv_err = pv_err.sum()
        pct_upb = total_pv_err / portfolio_upb * 100

        metrics_rows.append({
            'Bucket': bucket_label,
            'ME ($)': f'{me:+,.2f}',
            'MAE ($)': f'{mae:,.2f}',
            'RMSE ($)': f'{rmse:,.2f}',
            'Total PV Err ($M)': f'{total_pv_err/1e6:+,.2f}',
            '% of UPB': f'{pct_upb:+.3f}%',
        })

    metrics_df = pd.DataFrame(metrics_rows)
    print(metrics_df.to_string(index=False))

    # Cumulative PV error across all buckets
    total_all = sum(pv_errors[model_name][b].sum()
                    for b in BUCKET_LABELS if b in pv_errors[model_name])
    print(f'\n  Cumulative PV error: ${total_all/1e6:+,.2f}M '
          f'({total_all/portfolio_upb*100:+.3f}% of UPB)')

In [ ]:
# --- Comparison plots ---
colors = {'AJ': 'green', 'Cox': 'steelblue', 'RSF': 'darkorange', 'DeepHit': 'crimson'}
common_buckets = [b for b in BUCKET_LABELS if all(b in pv_errors[m] for m in MODEL_NAMES)]
x = np.arange(len(common_buckets))
width = 0.2

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. RMSE by bucket
ax = axes[0]
for j, m in enumerate(MODEL_NAMES):
    vals = [np.sqrt((pv_errors[m][b] ** 2).mean()) for b in common_buckets]
    ax.bar(x + j * width, vals, width, label=m, color=colors[m], alpha=0.8)
ax.set_title('PV RMSE by Bucket')
ax.set_xlabel('Forward bucket')
ax.set_ylabel('PV RMSE ($)')
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(common_buckets, rotation=45, ha='right', fontsize=8)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 2. ME (bias) by bucket
ax = axes[1]
for j, m in enumerate(MODEL_NAMES):
    vals = [pv_errors[m][b].mean() for b in common_buckets]
    ax.bar(x + j * width, vals, width, label=m, color=colors[m], alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.set_title('PV Mean Error (Bias) by Bucket')
ax.set_xlabel('Forward bucket')
ax.set_ylabel('PV ME ($)')
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(common_buckets, rotation=45, ha='right', fontsize=8)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 3. Total PV error as % of UPB (cumulative across buckets)
ax = axes[2]
for m in MODEL_NAMES:
    cum_pv = []
    running = 0.0
    for b in common_buckets:
        running += pv_errors[m][b].sum()
        cum_pv.append(running / portfolio_upb * 100)
    ax.plot(x + 1.5 * width, cum_pv, 'o-', color=colors[m], lw=2, label=m)
ax.axhline(0, color='black', lw=0.8)
ax.set_title('Cumulative PV Error (% of UPB)')
ax.set_xlabel('Forward bucket')
ax.set_ylabel('Cumulative PV error (% UPB)')
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(common_buckets, rotation=45, ha='right', fontsize=8)
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(f'Loan-Level PV Error by Bucket (cutoff {cutoff_str}, r = {DISCOUNT_RATE:.1%})',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'pv_error_by_bucket.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:

import pandas as pd

surv_df = pd.read_parquet('/Users/bartbronselaer/Developer/freddie-mac-survival-analysis/data/processed/survival_data_blumenstock.parquet')
print("Shape:", surv_df.shape)
print("\nColumns:")
for c in sorted(surv_df.columns):
    print(f"  {c}")


In [ ]:

# Get one row per loan to avoid weighting by number of observations
loan_level = surv_df.drop_duplicates(subset=['loan_sequence_number'])[['loan_sequence_number', 'orig_loan_term', 'vintage_year', 'orig_year_month']]

print(f"Unique loans: {len(loan_level):,}")
print(f"\nvintage_year range: {loan_level['vintage_year'].min()} - {loan_level['vintage_year'].max()}")
print(f"orig_loan_term values: {sorted(loan_level['orig_loan_term'].dropna().unique())}")

# Average orig_loan_term per vintage year
avg_term = loan_level.groupby('vintage_year')['orig_loan_term'].agg(['mean', 'median', 'count', 'std']).round(2)
avg_term.columns = ['Mean Term', 'Median Term', 'Loan Count', 'Std Dev']
avg_term.index.name = 'Vintage Year'

print("\n=== Average Orig Loan Term (months) by Vintage Year ===\n")
print(avg_term.to_string())

# Overall
print(f"\n--- Overall ---")
print(f"Mean:   {loan_level['orig_loan_term'].mean():.2f} months")
print(f"Median: {loan_level['orig_loan_term'].median():.0f} months")
print(f"Total loans: {len(loan_level):,}")
